# X-OPT - EXACT FACILITY REMOVAL REOPTIMIZATION


This notebook studies a facility-removal reoptimization scenario over the first ten OR-Library p-median instances. For each instance, the workflow is:

1. solve the original instance with the exact `python-mip` model and with the metaheuristic;
2. extract the Max-p-Cut, highest k-core, and densest subgraph from the metaheuristic long-term-memory co-occurrence graph;
3. take the exact optimal facility set, remove one facility from it, and reoptimize the exact model under four schemes:
   - baseline: keep the remaining `p - 1` exact facilities fixed and choose one replacement while forbidding the removed node;
   - highest k-core: keep the remaining `p - 1` exact facilities fixed and choose the replacement only from the highest k-core candidates;
   - densest subgraph: keep the remaining `p - 1` exact facilities fixed and choose the replacement only from the densest-subgraph candidates;
   - Max-p-Cut: for each facility in the exact optimal solution, create one replacement slot restricted to the Max-p-Cut group that contains that facility, while forbidding the removed node globally.

By default the removed facility is the first one in the sorted exact optimal solution, but the position is configurable.


### SETUP


In [ ]:
from __future__ import annotations

import gc
import os
import re
import sys
import math
import heapq

import numpy    as np
import pandas   as pd
import networkx as nx

from pathlib         import Path
from time            import perf_counter
from tqdm.auto       import tqdm
from itertools       import combinations
from IPython.display import display
from mip             import BINARY, Model, OptimizationStatus, minimize, xsum


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


In [ ]:
from lib.metrics import compute_solution_cost
from lib.explain import densest_subgraph_greedy
from lib.mip     import extract_open_facilities_candidates
from lib.utils   import as_sorted_tuple

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "instances").exists() and \
           (candidate / "pymedian" ).exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing 'instances' and 'pymedian'."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


In [ ]:
import pymedian


### CONFIGURATION


In [ ]:
def instance_sort_key(pathlike: str | Path) -> tuple[int, str]:
    stem  = Path(pathlike).stem
    match = re.search(r"(\d+)$", stem)

    if match is None:
        return (10**9, stem)

    return (int(match.group(1)), stem)


def canonical_instance_name(instance_name: str) -> str:
    instance_name = instance_name.strip()

    if instance_name.endswith(".txt"):
        return instance_name

    return f"{instance_name}.txt"


def parse_optional_int_env(name: str) -> int | None:
    raw_value = os.getenv(name)

    if raw_value is None:
        return None

    raw_value = raw_value.strip()
    if not raw_value:
        return None

    return int(raw_value)


def parse_optional_float_env(name: str) -> float | None:
    raw_value = os.getenv(name)

    if raw_value is None:
        return None

    raw_value = raw_value.strip()
    if not raw_value:
        return None

    return float(raw_value)


def list_orlibrary_instances(instances_dir: Path) -> list[str]:
    return sorted(
        [
            path.name
            for path in instances_dir.glob("pmed*.txt")
            if  path.name != "pmedopt.txt"
        ],
        key=instance_sort_key,
    )


def apply_instance_selection(
    instance_names: list[str],
    *,
    pattern: str | None = None,
    limit  : int | None = None,
) -> list[str]:
    selected = list(instance_names)

    if pattern:
        regex    = re.compile(pattern)
        selected = [name for name in selected if regex.search(name)]

    if limit is not None:
        selected = selected[:limit]

    return selected


def read_instance_metadata(instance_path: Path) -> dict[str, int]:
    header = instance_path.read_text().splitlines()[0].split()

    if len(header) < 3:
        raise ValueError(
            f"Could not parse instance header: {instance_path}"
        )

    return {
        "n": int(header[0]),
        "p": int(header[2]),
    }


def load_best_known_costs(pmedopt_path: Path) -> dict[str, float]:
    rows: dict[str, float] = {}

    for raw_line in pmedopt_path.read_text().splitlines()[1:]:
        line = raw_line.strip()
        if not line:
            continue

        instance_id, value = line.split()[:2]
        rows[instance_id]  = float(value)

    return rows


In [ ]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"
OUTPUT_DIR    = PROJECT_ROOT  / "notebooks" / "experiments_sbpo" / "artifacts"

RAW_RESULTS_CSV = OUTPUT_DIR / "facility_removal_reoptimization_raw.csv"
STRUCTURES_CSV  = OUTPUT_DIR / "facility_removal_reoptimization_structures.csv"
SUMMARY_CSV     = OUTPUT_DIR / "facility_removal_reoptimization_summary.csv"
FAILURES_CSV    = OUTPUT_DIR / "facility_removal_reoptimization_failures.csv"

INSTANCE_FILTER = os.getenv("FACREM_INSTANCE_FILTER")
MAX_INSTANCES   = parse_optional_int_env("FACREM_MAX_INSTANCES") or 10

RESTARTS       = parse_optional_int_env  ("FACREM_META_RESTARTS"      ) or 8
MAX_ITER       = parse_optional_int_env  ("FACREM_META_MAX_ITER"      ) or 25
FACTOR         = parse_optional_int_env  ("FACREM_META_FACTOR"        ) or 1
TOP_FRACTION   = parse_optional_float_env("FACREM_TOP_FRACTION"       ) or 0.10
DETAILS_FORMAT = os.getenv("FACREM_DETAILS_FORMAT", "binary")

EXACT_TIME_LIMIT_SECONDS = parse_optional_int_env("FACREM_EXACT_TIME_LIMIT_SECONDS") or 180
REOPT_TIME_LIMIT_SECONDS = parse_optional_int_env("FACREM_REOPT_TIME_LIMIT_SECONDS") or 180
GLOBAL_SEED              = parse_optional_int_env("FACREM_GLOBAL_SEED"             ) or 42
MAX_P_CUT_RESTARTS       = parse_optional_int_env("FACREM_MAX_P_CUT_RESTARTS"      ) or 30
MAX_P_CUT_MAX_ITER       = parse_optional_int_env("FACREM_MAX_P_CUT_MAX_ITER"      ) or 2000

REMOVED_FACILITY_POSITION = parse_optional_int_env("FACREM_REMOVED_FACILITY_POSITION") or 0

SAVE_RESULTS_CSV = os.getenv("FACREM_SAVE_RESULTS_CSV", "1") != "0"

if DETAILS_FORMAT != "binary":
    raise ValueError(
        "This notebook expects FACREM_DETAILS_FORMAT='binary' so the interpretability pipeline can build the co-occurrence matrix."
    )

ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)
BEST_KNOWN_COSTS = load_best_known_costs(PMEDOPT_PATH)

if not INSTANCE_NAMES:
    raise FileNotFoundError(
        f"No instances were selected from {INSTANCES_DIR}."
    )


In [ ]:
print(f"Instances discovered : {len(ALL_INSTANCE_NAMES)}")
print(f"Instances selected   : {len(INSTANCE_NAMES    )}")
print()
print("Instances to analyze:")
print(", ".join(Path(name).stem for name in INSTANCE_NAMES))
print()
print(f"Exact model time limit (s)      : {EXACT_TIME_LIMIT_SECONDS}")
print(f"Reoptimization time limit (s)   : {REOPT_TIME_LIMIT_SECONDS}")
print(f"Metaheuristic parameters        : restarts={RESTARTS:2d}, max_iter={MAX_ITER:4d}, factor={FACTOR}")
print(f"Max-p-Cut parameters            : restarts={MAX_P_CUT_RESTARTS:2d}, max_iter={MAX_P_CUT_MAX_ITER:4d}, seed={GLOBAL_SEED}")
print(f"Top fraction kept from the LTM  : {TOP_FRACTION:.0%}")
print(f"Removed facility position       : {REMOVED_FACILITY_POSITION}")
print()
print(f"Raw results CSV : {RAW_RESULTS_CSV}")
print(f"Structures CSV  : {STRUCTURES_CSV }")
print(f"Summary CSV     : {SUMMARY_CSV    }")


### GRAPH, EXACT MODEL AND INTERPRETABILITY HELPERS


In [ ]:
def read_orlibrary_graph(instance_path: Path) -> dict[str, object]:
    with instance_path.open() as file:
        header = file.readline().split()

        if len(header) < 3:
            raise ValueError(
                f"Could not parse instance header: {instance_path}"
            )

        n, m, p = map(int, header[:3])

        adjacency_map: list[dict[int, int]] = [dict() for _ in range(n)]
        raw_edge_count = 0

        for raw_line in file:
            line = raw_line.strip()
            if not line:
                continue

            u, v, cost = map(int, line.split())
            u -= 1
            v -= 1

            adjacency_map[u][v] = cost
            adjacency_map[v][u] = cost
            raw_edge_count += 1

    if raw_edge_count != m:
        raise ValueError(
            f"Expected {m} edges in {instance_path.name}, but found {raw_edge_count}."
        )

    adjacency = [
        sorted(neighbors.items())
        for neighbors in adjacency_map
    ]

    return {
        "n"        : n,
        "m"        : m,
        "p"        : p,
        "adjacency": adjacency,
    }


def all_pairs_shortest_paths(adjacency: list[list[tuple[int, int]]]) -> np.ndarray:
    n         = len(adjacency)
    distances = np.full((n, n), np.inf, dtype=np.float64)

    for source in range(n):
        row         = distances[source]
        row[source] = 0.0

        heap: list[tuple[float, int]] = [(0.0, source)]

        while heap:
            dist_u, u = heapq.heappop(heap)
            if dist_u > row[u]:
                continue

            for v, weight in adjacency[u]:
                candidate = dist_u + weight

                if candidate < row[v]:
                    row[v] = candidate
                    heapq.heappush(heap, (candidate, v))

        if np.isinf(row).any():
            raise ValueError("Instance graph is disconnected.")

    return distances.astype(np.int64)


def finite_or_none(value: float | int | None) -> float | None:
    if value is None:
        return None

    value = float(value)

    if not math.isfinite(value):
        return None

    return value


def gap_percent(value: float | int | None, reference: float | int | None) -> float:
    value     = finite_or_none(value    )
    reference = finite_or_none(reference)

    if value is None or reference is None or reference == 0.0:
        return np.nan

    return 100.0 * (value - reference) / reference





def format_facilities(values) -> str:
    facilities = as_sorted_tuple(values)

    if not facilities:
        return ""

    return " ".join(str(value + 1) for value in facilities)


def format_groups(groups) -> str:
    if not groups:
        return ""

    return " | ".join(format_facilities(group) for group in groups)


def choose_removed_facility(
    open_facilities: tuple[int, ...] | list[int],
    position       : int,
) -> int:
    facilities = as_sorted_tuple(open_facilities)

    if not facilities:
        raise ValueError("No facilities are available to remove.")

    if position < 0 or position >= len(facilities):
        raise ValueError(
            f"FACREM_REMOVED_FACILITY_POSITION={position} is outside the optimal solution size {len(facilities)}."
        )

    return facilities[position]


def find_group_containing(groups: list[tuple[int, ...]], facility: int) -> tuple[int, ...]:
    for group in groups:
        if facility in group:
            return tuple(sorted(group))

    raise ValueError(
        f"Facility {facility + 1} was not found in any Max-p-Cut group."
    )





In [ ]:
def build_top_ltm(
    long_term_memory: list[dict[str, object]],
    top_fraction    : float,
) -> tuple[list[dict[str, object]], np.ndarray, np.ndarray]:
    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    top_solution_count = max(
        1, int(np.ceil(len(long_term_memory) * top_fraction))
    )

    analysis_ltm = sorted(
        long_term_memory,
        key=lambda solution: float(solution["cost"]),
    )[:top_solution_count]

    matrix = np.vstack(
        [
            np.asarray(solution["facilities"], dtype=np.int8)
            for solution in analysis_ltm
        ]
    )
    costs = np.asarray(
        [
            float(solution["cost"])
            for solution in analysis_ltm
        ],
        dtype=float,
    )

    return analysis_ltm, matrix, costs


def build_cooccurrence_matrix(matrix: np.ndarray) -> np.ndarray:
    adjacency = np.asarray(matrix.T @ matrix, dtype=np.int64)

    np.fill_diagonal(adjacency, 0)

    return adjacency


def build_unweighted_graph(adjacency: np.ndarray) -> nx.Graph:
    graph = nx.Graph()
    graph.add_nodes_from(range(adjacency.shape[0]))

    rows, cols = np.where(np.triu(adjacency, k=1) > 0)

    graph.add_edges_from(
        (int(i), int(j))
        for i, j in zip(rows.tolist(), cols.tolist())
    )

    return graph


def total_edge_weight(adjacency: np.ndarray) -> float:
    return float(np.triu(adjacency, k=1).sum())


def total_edge_count(adjacency: np.ndarray) -> int:
    return int(np.count_nonzero(np.triu(adjacency, k=1) > 0))


def _random_labels(n: int, p: int, rng: np.random.Generator) -> np.ndarray:
    labels = rng.integers(0, p, size=n)

    if n >= p:
        permutation             = rng.permutation(n)
        labels[permutation[:p]] = np.arange(p)

    return labels


def _group_sums(adjacency: np.ndarray, labels: np.ndarray, p: int) -> np.ndarray:
    sums = np.zeros((adjacency.shape[0], p), dtype=float)

    for group_id in range(p):
        members = labels == group_id

        if np.any(members):
            sums[:, group_id] = adjacency[:, members].sum(axis=1)

    return sums


def max_p_cut_local_search(
    adjacency : np.ndarray,
    p         : int,
    *,
    n_restarts: int = 30,
    max_iter  : int = 2000,
    seed      : int = 42,
) -> tuple[np.ndarray, float, float]:
    adjacency = np.asarray(adjacency, dtype=float)

    n = adjacency.shape[0]
    p = max(1, min(int(p), n))

    total_weight = float(np.triu(adjacency, k=1).sum())
    if total_weight <= 1e-12:
        labels = np.arange(n, dtype=int) % p
        return labels, 0.0, 0.0

    rng = np.random.default_rng(seed)

    best_cut      = -np.inf
    best_internal =  np.inf
    best_labels   = None

    for _ in range(max(1, int(n_restarts))):
        labels = _random_labels(n, p, rng)
        sums   = _group_sums(adjacency, labels, p)

        internal = float(np.triu(adjacency * (labels[:, None] == labels[None, :]), k=1).sum())
        cut      = total_weight - internal

        for _ in range(max(1, int(max_iter))):
            improved = False

            for node in rng.permutation(n):
                old_group = labels[node]
                gains     = sums[node, old_group] - sums[node, :]
                gains[old_group] = -np.inf

                new_group = int(np.argmax(gains))
                gain      = float(gains[new_group])

                if gain > 1e-12:
                    labels[node] = new_group
                    cut += gain

                    sums[:, old_group] -= adjacency[:, node]
                    sums[:, new_group] += adjacency[:, node]

                    improved = True

            if not improved:
                break

        internal = total_weight - cut

        if cut > best_cut + 1e-12:
            best_cut      = float(cut)
            best_internal = float(internal)
            best_labels   = labels.copy()

    return best_labels, best_cut, best_internal


def best_facility_separation(labels: np.ndarray, best_set: set[int]) -> float:
    best_nodes = sorted(best_set)

    if len(best_nodes) <= 1:
        return 1.0

    separated_pairs = sum(
        labels[i] != labels[j]
        for i, j in combinations(best_nodes, 2)
    )

    return separated_pairs / math.comb(len(best_nodes), 2)


def extract_partition_groups(labels: np.ndarray, p: int) -> list[tuple[int, ...]]:
    return [
        tuple(sorted(np.flatnonzero(labels == group_id).tolist()))
        for group_id in range(p)
    ]


def extract_highest_k_core_nodes(graph: nx.Graph) -> tuple[dict[int, int], int, set[int]]:
    if graph.number_of_edges() == 0:
        core_numbers = {node: 0 for node in graph.nodes}
    else:
        core_numbers = nx.core_number(graph)

    max_core_level = max(core_numbers.values()) if core_numbers else 0

    highest_nodes = {
        node
        for node, level in core_numbers.items()
        if  level == max_core_level
    }

    return core_numbers, int(max_core_level), highest_nodes


def extract_structure_insights(
    summary: dict[str, object],
    details: dict[str, object],
    *,
    top_fraction       : float,
    max_p_cut_restarts : int,
    max_p_cut_max_iter : int,
    global_seed        : int,
) -> dict[str, object]:
    long_term_memory = details["long_term_memory"]

    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    analysis_ltm, matrix, costs = build_top_ltm(long_term_memory, top_fraction)
    adjacency                   = build_cooccurrence_matrix(matrix)
    unweighted_graph            = build_unweighted_graph(adjacency)

    best_facilities = tuple(
        sorted(
            int(value) - 1
            for value in summary["tspmed_facilities"]
        )
    )
    best_set = set(best_facilities)
    p        = int(summary["p"])
    n        = int(summary["n"])

    labels_maxcut, cut_weight, internal_weight = max_p_cut_local_search(
        adjacency,
        p,
        n_restarts=max_p_cut_restarts,
        max_iter  =max_p_cut_max_iter,
        seed      =global_seed,
    )
    max_p_cut_groups = extract_partition_groups(labels_maxcut, p)

    _, max_core_level, highest_k_core_nodes = extract_highest_k_core_nodes(unweighted_graph)
    densest_nodes, densest_density = densest_subgraph_greedy(
        adjacency,
        min_size=max(3, p),
    )

    return {
        "best_facilities"                  : best_facilities,
        "best_cost"                        : float(summary["tspmed_cost"]),
        "memory_size"                      : len(long_term_memory),
        "top_solution_count"               : len(analysis_ltm),
        "top_cost_cutoff"                  : float(costs.max()),
        "cooccurrence_edges"               : total_edge_count(adjacency),
        "cooccurrence_total_weight"        : total_edge_weight(adjacency),
        "max_p_cut_groups"                 : max_p_cut_groups,
        "max_p_cut_fraction_cut"           : cut_weight / max(1e-12, cut_weight + internal_weight),
        "max_p_cut_best_facility_separation": best_facility_separation(labels_maxcut, best_set),
        "k_core_max_level"                 : max_core_level,
        "highest_k_core_nodes"             : highest_k_core_nodes,
        "k_core_candidate_count"           : len(highest_k_core_nodes),
        "k_core_candidate_fraction"        : len(highest_k_core_nodes) / max(1, n),
        "k_core_best_set_recall"           : len(best_set.intersection(highest_k_core_nodes)) / max(1, len(best_set)),
        "densest_nodes"                    : densest_nodes,
        "densest_subgraph_density"         : densest_density,
        "densest_subgraph_candidate_count" : len(densest_nodes),
        "densest_subgraph_candidate_fraction": len(densest_nodes) / max(1, n),
        "densest_subgraph_best_set_recall" : len(best_set.intersection(densest_nodes)) / max(1, len(best_set)),
    }


### EXACT MODEL AND FACILITY-REMOVAL REOPTIMIZATION

In [ ]:
def build_pmedian_model(
    distances          : np.ndarray,
    p                  : int,
    *,
    allowed_facilities : tuple[int, ...] | list[int] | set[int] | None = None,
    fixed_open_facilities: tuple[int, ...] | list[int] | set[int] | None = None,
    forbidden_facilities: tuple[int, ...] | list[int] | set[int] | None = None,
) -> tuple[Model, list[list], list, list[int]]:
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError("Distances must be a square 2D array.")

    n = distances.shape[0]
    if not (1 <= p <= n):
        raise ValueError("P must satisfy 1 <= p <= n.")

    fixed_set     = set() if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set() if forbidden_facilities is None else {int(value) for value in forbidden_facilities}

    if not fixed_set.isdisjoint(forbidden_set):
        raise ValueError("A fixed-open facility cannot also be forbidden.")

    if allowed_facilities is None:
        candidate_set = set(range(n))
    else:
        candidate_set = {int(value) for value in allowed_facilities}

    candidate_set.difference_update(forbidden_set)
    candidate_set.update(fixed_set)

    candidate_facilities = sorted(candidate_set)

    if len(candidate_facilities) < p:
        raise ValueError(
            f"Candidate facility pool is too small: {len(candidate_facilities)} < {p}."
        )

    if len(fixed_set) > p:
        raise ValueError(
            f"Too many fixed facilities: {len(fixed_set)} > {p}."
        )

    model = Model(solver_name="CBC")
    model.verbose = 0

    y = [model.add_var(var_type=BINARY) for _ in candidate_facilities]
    x: list[list] = []

    for i in range(n):
        x_row = [
            model.add_var(var_type=BINARY)
            for _ in candidate_facilities
        ]
        x.append(x_row)

        model.add_constr(xsum(x_row) == 1)
        for pos in range(len(candidate_facilities)):
            model.add_constr(x_row[pos] <= y[pos])

    facility_to_pos = {
        facility: pos
        for pos, facility in enumerate(candidate_facilities)
    }

    for facility in fixed_set:
        if facility not in facility_to_pos:
            raise ValueError(
                f"Fixed facility {facility + 1} is not available in the candidate pool."
            )

        model.add_constr(y[facility_to_pos[facility]] == 1)

    model.add_constr(xsum(y) == p)

    model.objective = minimize(
        xsum(
            float(distances[i, facility]) * x[i][pos]
            for i in range(n)
            for pos, facility in enumerate(candidate_facilities)
        )
    )

    return model, x, y, candidate_facilities


def build_slot_group_pmedian_model(
    distances            : np.ndarray,
    slot_candidate_groups: list[tuple[int, ...]] | tuple[tuple[int, ...], ...],
) -> tuple[Model, list[list], list, list[int]]:
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError("Distances must be a square 2D array.")

    slot_groups = [
        as_sorted_tuple(group)
        for group in slot_candidate_groups
    ]
    p = len(slot_groups)
    n = distances.shape[0]

    if not slot_groups:
        raise ValueError("slot_candidate_groups cannot be empty.")

    if not (1 <= p <= n):
        raise ValueError("P must satisfy 1 <= number of slot groups <= n.")

    if any(not group for group in slot_groups):
        raise ValueError("Every slot candidate group must contain at least one facility.")

    candidate_facilities = sorted(
        {
            facility
            for group in slot_groups
            for facility in group
        }
    )

    if len(candidate_facilities) < p:
        raise ValueError(
            "The union of slot candidate groups is too small to open p distinct facilities: "
            f"{len(candidate_facilities)} < {p}."
        )

    model = Model(solver_name="CBC")
    model.verbose = 0

    y = [model.add_var(var_type=BINARY) for _ in candidate_facilities]
    x: list[list] = []

    for i in range(n):
        x_row = [
            model.add_var(var_type=BINARY)
            for _ in candidate_facilities
        ]
        x.append(x_row)

        model.add_constr(xsum(x_row) == 1)
        for pos in range(len(candidate_facilities)):
            model.add_constr(x_row[pos] <= y[pos])

    facility_to_pos = {
        facility: pos
        for pos, facility in enumerate(candidate_facilities)
    }
    pos_to_slot_vars = {
        pos: []
        for pos in range(len(candidate_facilities))
    }

    for group in slot_groups:
        slot_vars = []

        for facility in group:
            pos      = facility_to_pos[facility]
            slot_var = model.add_var(var_type=BINARY)

            slot_vars.append(slot_var)
            pos_to_slot_vars[pos].append(slot_var)

            model.add_constr(slot_var <= y[pos])

        model.add_constr(xsum(slot_vars) == 1)

    for slot_vars in pos_to_slot_vars.values():
        if slot_vars:
            model.add_constr(xsum(slot_vars) <= 1)

    model.add_constr(xsum(y) == p)

    model.objective = minimize(
        xsum(
            float(distances[i, facility]) * x[i][pos]
            for i in range(n)
            for pos, facility in enumerate(candidate_facilities)
        )
    )

    return model, x, y, candidate_facilities


def optimize_model(model: Model, time_limit_seconds: int | None) -> OptimizationStatus:
    if time_limit_seconds is None:
        return model.optimize()

    return model.optimize(max_seconds=float(time_limit_seconds))


def describe_solution_change(
    reference_open: tuple[int, ...] | list[int],
    candidate_open: tuple[int, ...] | list[int],
) -> dict[str, object]:
    reference_set = set(as_sorted_tuple(reference_open))
    candidate_set = set(as_sorted_tuple(candidate_open))

    overlap = sorted(reference_set.intersection(candidate_set))
    dropped = sorted(reference_set - candidate_set)
    added   = sorted(candidate_set - reference_set)

    return {
        "overlap_with_original_optimum_count"   : len(overlap),
        "overlap_with_original_optimum_fraction": len(overlap) / max(1, len(reference_set)),
        "dropped_optimal_facilities_count"      : len(dropped),
        "dropped_optimal_facilities"            : format_facilities(dropped),
        "new_facilities_count"                  : len(added),
        "new_facilities"                        : format_facilities(added),
    }


def solve_pmedian_variant(
    *,
    variant             : str,
    variant_order       : int,
    distances           : np.ndarray,
    p                   : int,
    time_limit_seconds  : int | None,
    allowed_facilities  : tuple[int, ...] | list[int] | set[int] | None = None,
    fixed_open_facilities: tuple[int, ...] | list[int] | set[int] | None = None,
    forbidden_facilities: tuple[int, ...] | list[int] | set[int] | None = None,
    slot_candidate_groups: list[tuple[int, ...]] | None = None,
) -> tuple[dict[str, object], tuple[int, ...]]:
    n = distances.shape[0]

    fixed_set     = set() if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set() if forbidden_facilities is None else {int(value) for value in forbidden_facilities}

    normalized_slot_groups = None

    if slot_candidate_groups is not None:
        normalized_slot_groups = [
            tuple(
                facility
                for facility in as_sorted_tuple(group)
                if  facility not in forbidden_set
            )
            for group in slot_candidate_groups
        ]
        candidate_set = {
            facility
            for group in normalized_slot_groups
            for facility in group
        }
    else:
        if allowed_facilities is None:
            candidate_set = set(range(n))
        else:
            candidate_set = {int(value) for value in allowed_facilities}

        candidate_set.difference_update(forbidden_set)
        candidate_set.update(fixed_set)

    candidate_facilities = sorted(candidate_set)

    row = {
        "variant"                    : variant,
        "variant_order"              : variant_order,
        "candidate_facility_count"   : len(candidate_facilities),
        "candidate_facility_fraction": len(candidate_facilities) / max(1, n),
        "fixed_facility_count"       : len(fixed_set),
        "fixed_facilities"           : format_facilities(fixed_set),
        "forbidden_facility_count"   : len(forbidden_set),
        "forbidden_facilities"       : format_facilities(forbidden_set),
        "slot_group_count"           : len(normalized_slot_groups) if normalized_slot_groups is not None else np.nan,
        "objective_value"            : np.nan,
        "objective_bound"            : np.nan,
        "solver_gap_percent"         : np.nan,
        "model_build_seconds"        : np.nan,
        "solve_seconds"              : np.nan,
        "total_variant_seconds"      : np.nan,
        "open_facilities_count"      : 0,
        "open_facilities"            : "",
        "status"                     : "ERROR",
        "error_message"              : None,
    }

    if len(candidate_facilities) < p:
        row.update(
            {
                "status"       : "INFEASIBLE_POOL",
                "error_message": f"Candidate facility pool smaller than p: {len(candidate_facilities)} < {p}.",
            }
        )
        return row, tuple()

    if normalized_slot_groups is not None and len(normalized_slot_groups) != p:
        row.update(
            {
                "status"       : "INVALID_SLOT_GROUPS",
                "error_message": (
                    "The number of slot candidate groups must match p: "
                    f"{len(normalized_slot_groups)} != {p}."
                ),
            }
        )
        return row, tuple()

    if normalized_slot_groups is not None and fixed_set:
        row.update(
            {
                "status"       : "INVALID_FIXED_SET",
                "error_message": "slot_candidate_groups cannot be combined with fixed_open_facilities in this model.",
            }
        )
        return row, tuple()

    if normalized_slot_groups is not None:
        empty_group_index = next(
            (
                idx
                for idx, group in enumerate(normalized_slot_groups, start=1)
                if  not group
            ),
            None,
        )

        if empty_group_index is not None:
            row.update(
                {
                    "status"       : "INFEASIBLE_POOL",
                    "error_message": (
                        "A Max-p-Cut slot candidate group became empty after "
                        f"forbidding facilities (slot {empty_group_index})."
                    ),
                }
            )
            return row, tuple()

    if len(fixed_set) > p:
        row.update(
            {
                "status"       : "INVALID_FIXED_SET",
                "error_message": f"Too many fixed facilities: {len(fixed_set)} > {p}.",
            }
        )
        return row, tuple()

    build_started_at = perf_counter()

    try:
        if normalized_slot_groups is None:
            model, x, y, candidate_facilities = build_pmedian_model(
                distances,
                p,
                allowed_facilities   =candidate_facilities,
                fixed_open_facilities=fixed_set,
                forbidden_facilities =forbidden_set,
            )
        else:
            model, x, y, candidate_facilities = build_slot_group_pmedian_model(
                distances,
                normalized_slot_groups,
            )
        build_seconds = perf_counter() - build_started_at

        solve_started_at = perf_counter()
        status           = optimize_model(model, time_limit_seconds)
        solve_seconds    = perf_counter() - solve_started_at

        has_incumbent = status in {
            OptimizationStatus.OPTIMAL,
            OptimizationStatus.FEASIBLE,
        }

        solver_objective = finite_or_none(model.objective_value if has_incumbent else None)

        open_facilities     = tuple()
        validated_objective = None

        if has_incumbent:
            open_facilities = extract_open_facilities_candidates(candidate_facilities, y)

            if len(open_facilities) != p:
                raise ValueError(
                    f"Expected {p} open facilities, but recovered {len(open_facilities)}."
                )

            validated_objective = compute_solution_cost(distances, open_facilities)
            if solver_objective is not None and validated_objective is not None:
                if abs(solver_objective - validated_objective) > 1e-4:
                    raise ValueError(
                        "Solver objective and validated objective do not match: "
                        f"{solver_objective} vs {validated_objective}."
                    )

        objective_value = validated_objective if validated_objective is not None else solver_objective

        row.update(
            {
                "objective_value"      : objective_value if objective_value is not None else np.nan,
                "model_build_seconds"  : build_seconds,
                "solve_seconds"        : solve_seconds,
                "total_variant_seconds": build_seconds + solve_seconds,
                "open_facilities_count": len(open_facilities),
                "open_facilities"      : format_facilities(open_facilities),
                "status"               : getattr(status, "name", str(status)),
                "error_message"        : None,
            }
        )

        return row, open_facilities
    except Exception as exc:
        row.update(
            {
                "status"             : "ERROR",
                "model_build_seconds": perf_counter() - build_started_at,
                "error_message"      : f"{type(exc).__name__}: {exc}",
            }
        )
        return row, tuple()
    finally:
        gc.collect()


### BENCHMARK EXECUTION


In [ ]:
VARIANT_SPECS = [
    {
        "variant"      : "baseline",
        "variant_order": 1,
        "allowed_key"  : None,
    },
    {
        "variant"      : "max_p_cut",
        "variant_order": 2,
        "allowed_key"  : None,
    },
    {
        "variant"      : "highest_k_core",
        "variant_order": 3,
        "allowed_key"  : "highest_k_core_nodes",
    },
    {
        "variant"      : "densest_subgraph",
        "variant_order": 4,
        "allowed_key"  : "densest_nodes",
    },
]


def build_pipeline_error_rows(
    instance_name  : str,
    instance_id    : str,
    metadata       : dict[str, int],
    best_known_cost: float | None,
    error_message  : str,
) -> list[dict[str, object]]:
    rows = []

    for spec in VARIANT_SPECS:
        rows.append(
            {
                "instance"       : instance_name,
                "instance_id"    : instance_id,
                "n"              : metadata.get("n", np.nan),
                "p"              : metadata.get("p", np.nan),
                "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,
                "variant"        : spec["variant"],
                "variant_order"  : spec["variant_order"],
                "status"         : "PIPELINE_ERROR",
                "error_message"  : error_message,
            }
        )

    return rows


def build_pipeline_error_structure_row(
    instance_name  : str,
    instance_id    : str,
    metadata       : dict[str, int],
    best_known_cost: float | None,
    error_message  : str,
) -> dict[str, object]:
    return {
        "instance"       : instance_name,
        "instance_id"    : instance_id,
        "n"              : metadata.get("n", np.nan),
        "p"              : metadata.get("p", np.nan),
        "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,
        "error_message"  : error_message,
    }


def run_single_instance_analysis(
    instance_name: str,
    *,
    restarts          : int,
    max_iter          : int,
    factor            : int,
    top_fraction      : float,
    details_format    : str,
    max_p_cut_restarts: int,
    max_p_cut_max_iter: int,
    global_seed       : int,
    exact_time_limit_seconds: int | None,
    reopt_time_limit_seconds: int | None,
    removed_facility_position: int,
) -> tuple[list[dict[str, object]], dict[str, object]]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    metadata        = read_instance_metadata(instance_path)
    best_known_cost = BEST_KNOWN_COSTS.get(instance_id)

    try:
        graph     = read_orlibrary_graph(instance_path)
        distances = all_pairs_shortest_paths(graph["adjacency"])
        p         = int(graph["p"])

        exact_row, exact_open = solve_pmedian_variant(
            variant            ="exact_reference",
            variant_order      =0,
            distances          =distances,
            p                  =p,
            time_limit_seconds =exact_time_limit_seconds,
        )

        if exact_row["status"] != "OPTIMAL":
            raise ValueError(
                "The original exact model must be solved to OPTIMAL for this analysis. "
                f"Received status={exact_row['status']}."
            )

        meta_started_at  = perf_counter()
        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =restarts,
            max_iter      =max_iter,
            factor        =factor,
            details_format=details_format,
        )
        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        structure_started_at = perf_counter()
        insights             = extract_structure_insights(
            summary,
            details,
            top_fraction       =top_fraction,
            max_p_cut_restarts =max_p_cut_restarts,
            max_p_cut_max_iter =max_p_cut_max_iter,
            global_seed        =global_seed,
        )
        structure_runtime_seconds = perf_counter() - structure_started_at

        removed_facility = choose_removed_facility(
            exact_open,
            removed_facility_position,
        )
        remaining_optimal = tuple(
            facility
            for facility in exact_open
            if  facility != removed_facility
        )

        removed_group = find_group_containing(
            insights["max_p_cut_groups"],
            removed_facility,
        )
        replacement_pool = tuple(
            value
            for value in removed_group
            if  value != removed_facility
        )
        max_p_cut_slot_groups = [
            find_group_containing(
                insights["max_p_cut_groups"],
                facility,
            )
            for facility in exact_open
        ]
        max_p_cut_slot_replacement_groups = [
            tuple(
                value
                for value in group
                if  value != removed_facility
            )
            for group in max_p_cut_slot_groups
        ]

        exact_objective = exact_row["objective_value"] if pd.notna(exact_row["objective_value"]) else None
        meta_best_set   = set(insights["best_facilities"])

        common = {
            "instance"       : instance_name,
            "instance_id"    : instance_id,
            "n"              : int(graph["n"]),
            "p"              : p,
            "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,

            "exact_original_status"              : exact_row["status"],
            "exact_original_objective"           : exact_row["objective_value"],
            "exact_original_bound"               : exact_row["objective_bound"],
            "exact_original_solver_gap_percent"  : exact_row["solver_gap_percent"],
            "exact_original_model_build_seconds" : exact_row["model_build_seconds"],
            "exact_original_solve_seconds"       : exact_row["solve_seconds"],
            "exact_original_total_seconds"       : exact_row["total_variant_seconds"],
            "exact_original_open_facilities"     : format_facilities(exact_open),
            "exact_original_gap_to_best_known_percent": gap_percent(exact_objective, best_known_cost),

            "metaheuristic_best_cost"                : insights["best_cost"],
            "metaheuristic_runtime_seconds"          : metaheuristic_runtime_seconds,
            "metaheuristic_gap_to_best_known_percent": gap_percent(insights["best_cost"], best_known_cost),
            "metaheuristic_gap_to_exact_percent"     : gap_percent(insights["best_cost"], exact_objective),
            "metaheuristic_best_facilities"          : format_facilities(insights["best_facilities"]),
            "metaheuristic_exact_overlap_count"      : len(set(exact_open).intersection(meta_best_set)),
            "metaheuristic_exact_overlap_fraction"   : len(set(exact_open).intersection(meta_best_set)) / max(1, p),

            "structure_extraction_runtime_seconds": structure_runtime_seconds,
            "pre_reoptimization_seconds"         : (
                exact_row["total_variant_seconds"]
                + metaheuristic_runtime_seconds
                + structure_runtime_seconds
            ),
            "memory_size"               : insights["memory_size"],
            "top_solution_count"        : insights["top_solution_count"],
            "top_cost_cutoff"           : insights["top_cost_cutoff"],
            "cooccurrence_edges"        : insights["cooccurrence_edges"],
            "cooccurrence_total_weight" : insights["cooccurrence_total_weight"],

            "max_p_cut_fraction_cut"             : insights["max_p_cut_fraction_cut"],
            "max_p_cut_best_facility_separation" : insights["max_p_cut_best_facility_separation"],
            "k_core_max_level"                   : insights["k_core_max_level"],
            "k_core_candidate_count"             : insights["k_core_candidate_count"],
            "k_core_candidate_fraction"          : insights["k_core_candidate_fraction"],
            "k_core_best_set_recall"             : insights["k_core_best_set_recall"],
            "densest_subgraph_candidate_count"   : insights["densest_subgraph_candidate_count"],
            "densest_subgraph_candidate_fraction": insights["densest_subgraph_candidate_fraction"],
            "densest_subgraph_density"           : insights["densest_subgraph_density"],
            "densest_subgraph_best_set_recall"   : insights["densest_subgraph_best_set_recall"],

            "removed_facility"                : removed_facility + 1,
            "removed_facility_zero_based"     : removed_facility,
            "removed_facility_position"       : removed_facility_position,
            "remaining_optimal_facilities"    : format_facilities(remaining_optimal),
            "remaining_optimal_facilities_count": len(remaining_optimal),
            "max_p_cut_optimal_slot_group_count": len(max_p_cut_slot_groups),
            "max_p_cut_unique_optimal_group_count": len({tuple(group) for group in max_p_cut_slot_groups}),
            "max_p_cut_removed_group_size"    : len(removed_group),
            "max_p_cut_replacement_pool_size" : len(replacement_pool),
        }

        structure_row = {
            "instance"       : instance_name,
            "instance_id"    : instance_id,
            "n"              : int(graph["n"]),
            "p"              : p,
            "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,
            "exact_original_objective"      : exact_row["objective_value"],
            "exact_original_open_facilities": format_facilities(exact_open),
            "metaheuristic_best_cost"       : insights["best_cost"],
            "metaheuristic_best_facilities" : format_facilities(insights["best_facilities"]),
            "removed_facility"              : removed_facility + 1,
            "removed_facility_position"     : removed_facility_position,
            "remaining_optimal_facilities"  : format_facilities(remaining_optimal),
            "removed_facility_in_meta_best" : int(removed_facility in meta_best_set),
            "max_p_cut_groups"              : format_groups(insights["max_p_cut_groups"]),
            "max_p_cut_removed_group"       : format_facilities(removed_group),
            "max_p_cut_replacement_pool"    : format_facilities(replacement_pool),
            "max_p_cut_optimal_slot_groups" : format_groups(max_p_cut_slot_groups),
            "max_p_cut_slot_replacement_groups": format_groups(max_p_cut_slot_replacement_groups),
            "highest_k_core_nodes"          : format_facilities(insights["highest_k_core_nodes"]),
            "densest_nodes"                 : format_facilities(insights["densest_nodes"]),
            "error_message"                 : None,
        }

        rows: list[dict[str, object]] = []

        for spec in VARIANT_SPECS:
            allowed_facilities    = None
            fixed_open_facilities = remaining_optimal
            slot_candidate_groups = None

            if spec["variant"] == "max_p_cut":
                fixed_open_facilities = None
                slot_candidate_groups = list(max_p_cut_slot_replacement_groups)
            elif spec["allowed_key"] is not None:
                allowed_facilities = tuple(sorted(insights[spec["allowed_key"]]))

            row, open_facilities = solve_pmedian_variant(
                variant             =spec["variant"],
                variant_order       =spec["variant_order"],
                distances           =distances,
                p                   =p,
                time_limit_seconds  =reopt_time_limit_seconds,
                allowed_facilities  =allowed_facilities,
                fixed_open_facilities=fixed_open_facilities,
                forbidden_facilities=(removed_facility,),
                slot_candidate_groups=slot_candidate_groups,
            )

            row.update(common)
            row.update(describe_solution_change(exact_open, open_facilities))
            row["degradation_vs_exact_percent"] = gap_percent(
                row["objective_value"] if pd.notna(row["objective_value"]) else None,
                exact_objective,
            )
            row["total_pipeline_seconds"] = common["pre_reoptimization_seconds"] + (
                row["total_variant_seconds"] if pd.notna(row["total_variant_seconds"]) else 0.0
            )

            rows.append(row)

        return rows, structure_row
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        return (
            build_pipeline_error_rows(
                instance_name,
                instance_id,
                metadata,
                best_known_cost,
                error_message,
            ),
            build_pipeline_error_structure_row(
                instance_name,
                instance_id,
                metadata,
                best_known_cost,
                error_message,
            ),
        )


def sort_by_instance(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "instance" not in df.columns:
        return df

    ordered = df.copy()
    ordered["instance_order"] = ordered["instance"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        ordered.sort_values(["instance_order", "instance"])
               .drop(columns="instance_order")
               .reset_index(drop=True)
    )


def run_benchmark(
    instance_names: list[str],
    *,
    restarts          : int,
    max_iter          : int,
    factor            : int,
    top_fraction      : float,
    details_format    : str,
    max_p_cut_restarts: int,
    max_p_cut_max_iter: int,
    global_seed       : int,
    exact_time_limit_seconds: int | None,
    reopt_time_limit_seconds: int | None,
    removed_facility_position: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not instance_names:
        raise ValueError("The benchmark requires at least one instance.")

    rows: list[dict[str, object]] = []
    structure_rows: list[dict[str, object]] = []

    for instance_name in tqdm(
        instance_names,
        total        =len(instance_names),
        desc         ="Facility-removal benchmark",
        dynamic_ncols=True,
    ):
        instance_rows, structure_row = run_single_instance_analysis(
            instance_name,
            restarts          =restarts,
            max_iter          =max_iter,
            factor            =factor,
            top_fraction      =top_fraction,
            details_format    =details_format,
            max_p_cut_restarts=max_p_cut_restarts,
            max_p_cut_max_iter=max_p_cut_max_iter,
            global_seed       =global_seed,
            exact_time_limit_seconds=exact_time_limit_seconds,
            reopt_time_limit_seconds=reopt_time_limit_seconds,
            removed_facility_position=removed_facility_position,
        )
        rows.extend(instance_rows)
        structure_rows.append(structure_row)

    results_df = pd.DataFrame(rows)
    structures_df = pd.DataFrame(structure_rows)

    if not results_df.empty:
        results_df["variant_order"] = pd.to_numeric(
            results_df["variant_order"],
            errors="coerce",
        )
        results_df = sort_by_instance(results_df)
        results_df = results_df.sort_values(
            ["instance", "variant_order"],
            kind="stable",
        ).reset_index(drop=True)

    structures_df = sort_by_instance(structures_df)

    return results_df, structures_df


def add_forbidden_baseline_reference(results_df: pd.DataFrame) -> pd.DataFrame:
    merged = results_df.copy()

    if merged.empty:
        return merged

    if "objective_value" not in merged.columns:
        merged["objective_value"] = np.nan
    if "status" not in merged.columns:
        merged["status"] = np.nan

    baseline_df = (
        merged.loc[
            merged["variant"] == "baseline",
            ["instance", "status", "objective_value"],
        ]
        .rename(
            columns={
                "status"         : "baseline_status",
                "objective_value": "baseline_objective_value",
            }
        )
    )

    merged = merged.merge(baseline_df, on="instance", how="left")

    merged["gap_to_forbidden_baseline_percent"] = np.where(
        merged["objective_value"].notna() & merged["baseline_objective_value"].notna(),
        100.0 * (merged["objective_value"] - merged["baseline_objective_value"]) / merged["baseline_objective_value"],
        np.nan,
    )
    merged["baseline_is_optimal"] = merged["baseline_status"].eq("OPTIMAL")

    return merged


def build_wide_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return results_df.copy()

    instance_columns = [
        "instance",
        "instance_id",
        "n",
        "p",
        "best_known_cost",
        "exact_original_status",
        "exact_original_objective",
        "exact_original_gap_to_best_known_percent",
        "exact_original_total_seconds",
        "exact_original_open_facilities",
        "metaheuristic_best_cost",
        "metaheuristic_gap_to_best_known_percent",
        "metaheuristic_gap_to_exact_percent",
        "metaheuristic_runtime_seconds",
        "metaheuristic_best_facilities",
        "metaheuristic_exact_overlap_fraction",
        "structure_extraction_runtime_seconds",
        "pre_reoptimization_seconds",
        "memory_size",
        "top_solution_count",
        "top_cost_cutoff",
        "cooccurrence_edges",
        "cooccurrence_total_weight",
        "max_p_cut_fraction_cut",
        "max_p_cut_best_facility_separation",
        "k_core_max_level",
        "k_core_candidate_count",
        "k_core_candidate_fraction",
        "densest_subgraph_candidate_count",
        "densest_subgraph_candidate_fraction",
        "densest_subgraph_density",
        "removed_facility",
        "removed_facility_position",
        "remaining_optimal_facilities",
        "max_p_cut_optimal_slot_group_count",
        "max_p_cut_unique_optimal_group_count",
        "max_p_cut_removed_group_size",
        "max_p_cut_replacement_pool_size",
    ]

    metric_columns = [
        "status",
        "candidate_facility_count",
        "fixed_facility_count",
        "slot_group_count",
        "objective_value",
        "objective_bound",
        "solver_gap_percent",
        "degradation_vs_exact_percent",
        "gap_to_forbidden_baseline_percent",
        "overlap_with_original_optimum_fraction",
        "new_facilities_count",
        "model_build_seconds",
        "solve_seconds",
        "total_variant_seconds",
        "total_pipeline_seconds",
    ]

    working = results_df.copy()
    for column in [*instance_columns, *metric_columns]:
        if column not in working.columns:
            working[column] = np.nan

    base_df = working[instance_columns].drop_duplicates("instance")

    wide_df = working.pivot(
        index  ="instance",
        columns="variant",
        values =metric_columns,
    )
    wide_df.columns = [f"{variant}__{metric}" for metric, variant in wide_df.columns]
    wide_df = wide_df.reset_index()

    return base_df.merge(wide_df, on="instance", how="left")


def build_variant_aggregate(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    working = results_df.copy()
    for column in [
        "objective_value",
        "candidate_facility_fraction",
        "total_variant_seconds",
        "total_pipeline_seconds",
        "degradation_vs_exact_percent",
        "gap_to_forbidden_baseline_percent",
        "solver_gap_percent",
    ]:
        if column not in working.columns:
            working[column] = np.nan

    if "status" not in working.columns:
        working["status"] = np.nan

    return (
        working.groupby("variant", dropna=False)
               .agg(
                   instances                    =("instance", "nunique"),
                   solved_instances             =("objective_value", lambda s: int(s.notna().sum())),
                   optimal_instances            =("status", lambda s: int((s == "OPTIMAL").sum())),
                   mean_candidate_fraction      =("candidate_facility_fraction", "mean"),
                   mean_total_variant_seconds   =("total_variant_seconds", "mean"),
                   mean_total_pipeline_seconds  =("total_pipeline_seconds", "mean"),
                   mean_degradation_vs_exact_pct=("degradation_vs_exact_percent", "mean"),
                   mean_gap_to_baseline_pct     =("gap_to_forbidden_baseline_percent", "mean"),
                   mean_solver_gap_pct          =("solver_gap_percent", "mean"),
               )
               .reset_index()
               .sort_values("variant")
               .reset_index(drop=True)
    )


### APPLY


In [ ]:
raw_results_df, structures_df = run_benchmark(
    INSTANCE_NAMES,
    restarts          =RESTARTS,
    max_iter          =MAX_ITER,
    factor            =FACTOR,
    top_fraction      =TOP_FRACTION,
    details_format    =DETAILS_FORMAT,
    max_p_cut_restarts=MAX_P_CUT_RESTARTS,
    max_p_cut_max_iter=MAX_P_CUT_MAX_ITER,
    global_seed       =GLOBAL_SEED,
    exact_time_limit_seconds=EXACT_TIME_LIMIT_SECONDS,
    reopt_time_limit_seconds=REOPT_TIME_LIMIT_SECONDS,
    removed_facility_position=REMOVED_FACILITY_POSITION,
)

results_df           = add_forbidden_baseline_reference(raw_results_df)
wide_summary_df      = build_wide_summary(results_df)
variant_aggregate_df = build_variant_aggregate(results_df)
failures_df          = results_df[results_df["error_message"].notna()].copy()


In [ ]:
display(variant_aggregate_df.round(4))


In [ ]:
display(structures_df.head())


In [ ]:
display(wide_summary_df.head().round(4))


In [ ]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(RAW_RESULTS_CSV, index=False)
    structures_df.to_csv(STRUCTURES_CSV, index=False)
    wide_summary_df.to_csv(SUMMARY_CSV, index=False)
    failures_df.to_csv(FAILURES_CSV, index=False)

    print(f"Raw results saved to : {RAW_RESULTS_CSV}")
    print(f"Structures saved to  : {STRUCTURES_CSV }")
    print(f"Summary saved to     : {SUMMARY_CSV    }")
    print(f"Failures saved to    : {FAILURES_CSV   }")
